# Titanic — EDA, Feature Engineering, and Ensemble Modeling

Study notebook for the Kaggle Titanic dataset. It walks through exploratory
data analysis, feature engineering, a multi-metric comparison of tree-based
classifiers, Bayesian hyperparameter tuning with **Optuna**, a **Stacking**
ensemble, and model interpretation with **SHAP** and feature importances.

**Pipeline summary**
1. Load & explore the raw data.
2. Engineer features (titles, family size, deck, fare-per-person, etc.).
3. Compare Random Forest, Extra Trees, XGBoost, LightGBM, and CatBoost with
   `cross_validate` across accuracy, precision, recall, F1, and ROC AUC.
4. Tune the top candidates with Optuna.
5. Stack the tuned models with a `StackingClassifier`.
6. Train the final model, generate the submission file, and explain
   predictions with feature importance and SHAP.

## 1. Imports, paths, and global settings

In [2]:
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = [
    "pandas",
    "numpy",
    "seaborn",
    "matplotlib",
    "scikit-learn",
    "jupyter",
    "xgboost",
    "lightgbm",
    "catboost",
    "optuna",
    "shap",
    "ipykernel",
]

for package in REQUIRED_PACKAGES:
    if importlib.util.find_spec(package) is None:
        print(f"Installing missing package: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import optuna
import optuna.visualization.matplotlib as optuna_viz


from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_SPLITS = 10
N_OPTUNA_TRIALS = 40
STACK_TOP_N = 3
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", None)

PALETTE = {"Did not survive": "#d95f5f", "Survived": "#2f9e78"}
ACCENT = "#4c78a8"
ACCENT_2 = "#6f7db8"
SURVIVAL_ORDER = ["Did not survive", "Survived"]

cwd = Path.cwd()

candidate_dirs = [cwd, cwd.parent]
for candidate in candidate_dirs:
    if (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
        DATA_DIR = candidate
        PROJECT_DIR = candidate / "Titanic" if (candidate / "Titanic").exists() else candidate
        break
else:
    raise FileNotFoundError("Could not find train.csv and test.csv in the current folder or parent folder.")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

TRAIN_PATH, TEST_PATH, SUBMISSION_PATH


Installing missing package: scikit-learn


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Data loading

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

train.head()

In [ ]:
test.head()

## 3. Overview

In [ ]:
train.info()

In [ ]:
train.describe(include="all")

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing[missing > 0].to_frame("missing_values")

## 4. Feature engineering

In [ ]:
def extract_title(name: str) -> str:
    match = re.search(r",\s*([^\.]+)\.", name)
    if not match:
        return "Unknown"

    title = match.group(1).strip()
    title_map = {
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
        "Lady": "Royalty", "Countess": "Royalty", "Sir": "Royalty",
        "Jonkheer": "Royalty", "Don": "Royalty", "Dona": "Royalty",
        "Capt": "Officer", "Col": "Officer", "Major": "Officer",
        "Dr": "Officer", "Rev": "Officer",
    }
    return title_map.get(title, title)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Feature set shared by the EDA and the modeling stage (mirrors
    `prepare_features` in titanic_model.py)."""
    df = df.copy()
    if "Survived" in df.columns:
        df["SurvivalLabel"] = df["Survived"].map({0: "Did not survive", 1: "Survived"})

    df["Title"] = df["Name"].apply(extract_title)
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["FamilyGroup"] = pd.cut(df["FamilySize"], bins=[0, 1, 4, 20], labels=["Alone", "Small", "Large"])

    df["Deck"] = df["Cabin"].fillna("Unknown").astype(str).str[0]
    df["Deck"] = df["Deck"].replace("U", "Unknown")
    df["CabinKnown"] = df["Cabin"].notna().astype(int)

    ticket_counts = df["Ticket"].value_counts()
    df["TicketGroup"] = df["Ticket"].map(ticket_counts)
    df["TicketGroup"] = pd.cut(df["TicketGroup"], bins=[0, 1, 4, 20], labels=["Single", "Small", "Large"])

    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    df["AgeClass"] = df["Age"] * df["Pclass"]
    df["Child"] = (df["Age"] < 16).astype(int)
    df["Mother"] = ((df["Sex"] == "female") & (df["Parch"] > 0) & (df["Age"] > 18) & (df["Title"] != "Miss")).astype(int)
    return df


train_eda = add_features(train)
test_eda = add_features(test)

train_eda[["Name", "Title", "FamilySize", "IsAlone", "Deck", "FarePerPerson", "Child", "Mother"]].head()

## 5. Exploratory data analysis

In [ ]:
def annotate_rate_bars(ax, y_offset: float = 0.02) -> None:
    for patch in ax.patches:
        height = patch.get_height()
        if pd.notna(height):
            ax.text(
                patch.get_x() + patch.get_width() / 2, height + y_offset,
                f"{height:.0%}", ha="center", va="bottom", fontsize=9, fontweight="bold",
            )


def format_rate_axis(ax, title: str, xlabel: str = "") -> None:
    ax.set_title(title, pad=12, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Survival rate")
    ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.grid(axis="y", alpha=0.25)


survival_summary = (
    train_eda["SurvivalLabel"].value_counts(normalize=True).reindex(SURVIVAL_ORDER)
    .rename("Share").reset_index().rename(columns={"SurvivalLabel": "Survival"})
)

survival_rate = train_eda["Survived"].mean()
print(f"Overall survival rate: {survival_rate:.2%}")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=survival_summary, x="Survival", y="Share", hue="Survival",
            order=SURVIVAL_ORDER, palette=PALETTE, legend=False, ax=ax)
ax.set_title("Target distribution", pad=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Passenger share")
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
annotate_rate_bars(ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

rate_specs = [
    ("Sex", "Survival by sex", None, 0),
    ("Pclass", "Survival by passenger class", [1, 2, 3], 0),
    ("Embarked", "Survival by embarkation port", None, 0),
    ("FamilySize", "Survival by family size", sorted(train_eda["FamilySize"].unique()), 0),
]

for ax, (feature, title, order, rotation) in zip(axes.flat, rate_specs):
    sns.barplot(data=train_eda, x=feature, y="Survived", order=order,
                errorbar=None, color=ACCENT, ax=ax)
    format_rate_axis(ax, title, feature)
    ax.tick_params(axis="x", rotation=rotation)
    annotate_rate_bars(ax, y_offset=0.015)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=train_eda, x="Age", hue="SurvivalLabel", hue_order=SURVIVAL_ORDER,
             multiple="layer", stat="density", common_norm=False, bins=28,
             alpha=0.45, palette=PALETTE, ax=axes[0])
axes[0].set_title("Age distribution by survival", pad=12, fontweight="bold")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")
axes[0].grid(axis="y", alpha=0.25)

sns.boxplot(data=train_eda, x="SurvivalLabel", y="Fare", hue="SurvivalLabel",
            order=SURVIVAL_ORDER, palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_title("Fare distribution by survival", pad=12, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Fare")
axes[1].set_ylim(0, train_eda["Fare"].quantile(0.98))
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

title_order = train_eda.groupby("Title")["Survived"].mean().sort_values(ascending=False).index
deck_order = train_eda.groupby("Deck")["Survived"].mean().sort_values(ascending=False).index

sns.barplot(data=train_eda, x="Title", y="Survived", order=title_order,
            errorbar=None, color=ACCENT, ax=axes[0])
format_rate_axis(axes[0], "Survival by extracted title", "Title")
axes[0].tick_params(axis="x", rotation=35)
annotate_rate_bars(axes[0], y_offset=0.015)

sns.barplot(data=train_eda, x="Deck", y="Survived", order=deck_order,
            errorbar=None, color=ACCENT_2, ax=axes[1])
format_rate_axis(axes[1], "Survival by cabin deck", "Deck")
annotate_rate_bars(axes[1], y_offset=0.015)

plt.tight_layout()
plt.show()

### 5.1 New engineered features

A closer look at the extra features used by the model: family/ticket
grouping, whether the cabin is known, being a child or a mother, and the
fare paid per person (rather than per ticket).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

sns.barplot(data=train_eda, x="FamilyGroup", y="Survived", order=["Alone", "Small", "Large"],
            errorbar=None, color=ACCENT, ax=axes[0, 0])
format_rate_axis(axes[0, 0], "Survival by family group", "Family group")
annotate_rate_bars(axes[0, 0], y_offset=0.015)

sns.barplot(data=train_eda, x="TicketGroup", y="Survived", order=["Single", "Small", "Large"],
            errorbar=None, color=ACCENT_2, ax=axes[0, 1])
format_rate_axis(axes[0, 1], "Survival by ticket group", "Ticket group")
annotate_rate_bars(axes[0, 1], y_offset=0.015)

sns.barplot(data=train_eda, x="CabinKnown", y="Survived", order=[0, 1],
            errorbar=None, color=ACCENT, ax=axes[0, 2])
format_rate_axis(axes[0, 2], "Survival by cabin known", "Cabin known (0/1)")
annotate_rate_bars(axes[0, 2], y_offset=0.015)

sns.barplot(data=train_eda, x="Child", y="Survived", order=[0, 1],
            errorbar=None, color=ACCENT_2, ax=axes[1, 0])
format_rate_axis(axes[1, 0], "Survival: child vs adult", "Child (age < 16)")
annotate_rate_bars(axes[1, 0], y_offset=0.015)

sns.barplot(data=train_eda, x="Mother", y="Survived", order=[0, 1],
            errorbar=None, color=ACCENT, ax=axes[1, 1])
format_rate_axis(axes[1, 1], "Survival: mother flag", "Mother (0/1)")
annotate_rate_bars(axes[1, 1], y_offset=0.015)

sns.kdeplot(data=train_eda, x="FarePerPerson", hue="SurvivalLabel", hue_order=SURVIVAL_ORDER,
            common_norm=False, fill=True, alpha=0.35, palette=PALETTE, ax=axes[1, 2])
axes[1, 2].set_xlim(0, train_eda["FarePerPerson"].quantile(0.98))
axes[1, 2].set_title("Fare per person by survival", pad=12, fontweight="bold")
axes[1, 2].set_xlabel("Fare per person")
axes[1, 2].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
numeric_corr = train_eda[
    ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize",
     "IsAlone", "FarePerPerson", "AgeClass", "CabinKnown", "Child", "Mother"]
].corr()

plt.figure(figsize=(9.5, 7.5))
ax = sns.heatmap(numeric_corr, annot=True, fmt=".2f", cmap="vlag", center=0,
                  square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation matrix (engineered numeric features)", pad=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Modeling preparation

In [ ]:
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_features(df)

    df["Age"] = df["Age"].fillna(df.groupby(["Title", "Sex", "Pclass"])["Age"].transform("median"))
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna("S")

    selected = [
        "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title",
        "FamilySize", "IsAlone", "Deck", "CabinKnown", "FarePerPerson",
        "AgeClass", "Child", "Mother", "FamilyGroup", "TicketGroup",
    ]
    return df[selected]


X = prepare_features(train)
y = train["Survived"]
X_test = prepare_features(test)

X.head()

## 7. Preprocessing pipeline

All five candidate models are tree-based, so a single preprocessor
(median/most-frequent imputation + one-hot encoding, no scaling) is used for
every model — no separate linear-model branch is needed anymore.

In [ ]:
NUMERIC_FEATURES = [
    "Age", "Fare", "Pclass", "SibSp", "Parch", "FamilySize",
    "FarePerPerson", "AgeClass", "Child", "Mother", "CabinKnown", "IsAlone",
]
CATEGORICAL_FEATURES = ["Sex", "Embarked", "Title", "Deck", "FamilyGroup", "TicketGroup"]


def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ])


def build_pipeline(model) -> Pipeline:
    return Pipeline([("preprocessor", build_preprocessor()), ("model", model)])

## 8. Baseline model comparison

Five tree-based classifiers — Random Forest, Extra Trees, XGBoost, LightGBM,
CatBoost — are compared with `cross_validate` across five metrics at once
(accuracy, precision, recall, F1, ROC AUC), instead of the single-metric
`cross_val_score` used previously. AdaBoost and Gradient Boosting were
dropped as redundant with the boosting libraries already in the mix.

In [ ]:
def build_models() -> dict:
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=2, max_features="sqrt",
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=2, max_features="sqrt",
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8,
            colsample_bytree=0.8, random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=500, learning_rate=0.03, num_leaves=31,
            random_state=RANDOM_STATE, verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=500, learning_rate=0.03, depth=6, loss_function="Logloss",
            verbose=False, random_state=RANDOM_STATE,
        ),
    }


cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

rows = []
for model_name, model in build_models().items():
    pipeline = build_pipeline(model)
    cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {"model": model_name}
    for metric in SCORING:
        scores = cv_results[f"test_{metric}"]
        row[f"{metric}_mean"] = scores.mean()
        row[f"{metric}_std"] = scores.std()
    rows.append(row)

baseline_results = pd.DataFrame(rows).sort_values("accuracy_mean", ascending=False).reset_index(drop=True)
baseline_results

In [ ]:
metric_cols = [f"{m}_mean" for m in SCORING]
plot_df = baseline_results.set_index("model")[metric_cols]
plot_df.columns = [c.replace("_mean", "") for c in plot_df.columns]

fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df.plot(kind="bar", ax=ax, width=0.78,
             color=["#4c78a8", "#72b7b2", "#e45756", "#f2b701", "#6f7db8"])
ax.set_title("Cross-validated metrics by model", pad=12, fontweight="bold")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1.0)
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

best_baseline_name = baseline_results.loc[0, "model"]
print(f"Best baseline model (by accuracy): {best_baseline_name}")

## 9. Hyperparameter tuning with Optuna

The top `STACK_TOP_N` baseline models are tuned with Optuna's Bayesian
(TPE) search, maximizing mean 5-fold CV accuracy.

In [ ]:
def _suggest_params(trial: optuna.Trial, model_name: str) -> dict:
    if model_name == "Random Forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 6),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
    if model_name == "Extra Trees":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 6),
        }
    if model_name == "XGBoost":
        return {
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        }
    if model_name == "LightGBM":
        return {
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 900, step=100),
            "num_leaves": trial.suggest_int("num_leaves", 8, 64),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
        }
    if model_name == "CatBoost":
        return {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "iterations": trial.suggest_int("iterations", 200, 900, step=100),
        }
    raise ValueError(f"Unknown model: {model_name}")


def _build_model_from_params(model_name: str, params: dict):
    if model_name == "Random Forest":
        return RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, **params)
    if model_name == "Extra Trees":
        return ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, **params)
    if model_name == "XGBoost":
        return XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1, **params)
    if model_name == "LightGBM":
        return LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, **params)
    if model_name == "CatBoost":
        return CatBoostClassifier(loss_function="Logloss", verbose=False, random_state=RANDOM_STATE, **params)
    raise ValueError(f"Unknown model: {model_name}")


def tune_model_optuna(model_name: str, X_train: pd.DataFrame, y_train: pd.Series, n_trials: int = N_OPTUNA_TRIALS):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial: optuna.Trial) -> float:
        params = _suggest_params(trial, model_name)
        model = _build_model_from_params(model_name, params)
        pipeline = build_pipeline(model)
        scores = cross_val_score(pipeline, X_train, y_train, cv=inner_cv, scoring="accuracy", n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=f"{model_name}_optuna")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_model = _build_model_from_params(model_name, study.best_params)
    return best_model, study


top_models = baseline_results["model"].head(STACK_TOP_N).tolist()
print(f"Models selected for tuning + stacking: {top_models}")

tuned_models = {}
studies = {}
for model_name in top_models:
    best_model, study = tune_model_optuna(model_name, X, y)
    tuned_models[model_name] = best_model
    studies[model_name] = study
    print(f"{model_name:15s} best CV accuracy = {study.best_value:.4f}")

In [ ]:
best_tuned_name = max(studies, key=lambda name: studies[name].best_value)
best_study = studies[best_tuned_name]

ax1 = optuna_viz.plot_optimization_history(best_study)
ax1.set_title(f"Optuna optimization history — {best_tuned_name}", fontweight="bold")
ax1.figure.set_size_inches(9, 4.5)
plt.tight_layout()
plt.show()

ax2 = optuna_viz.plot_param_importances(best_study)
ax2.set_title(f"Hyperparameter importance — {best_tuned_name}", fontweight="bold")
ax2.figure.set_size_inches(8, 4.5)
plt.tight_layout()
plt.show()

## 10. Stacking ensemble

The tuned top models are combined into a `StackingClassifier` (replacing the
previous `VotingClassifier`), with a logistic-regression meta-learner
trained on their out-of-fold predictions.

In [ ]:
def build_stacking_model(models: dict) -> StackingClassifier:
    estimators = [(name.lower().replace(" ", "_"), clone(model)) for name, model in models.items()]
    return StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        cv=5,
        n_jobs=-1,
    )


stacking_model = build_stacking_model(tuned_models)
stacking_pipeline = build_pipeline(stacking_model)

stack_cv = cross_validate(stacking_pipeline, X, y, cv=cv, scoring=SCORING, n_jobs=-1)

comparison_rows = [{
    "model": "Stacking Ensemble",
    **{f"{m}_mean": stack_cv[f"test_{m}"].mean() for m in SCORING},
    **{f"{m}_std": stack_cv[f"test_{m}"].std() for m in SCORING},
}]
comparison = pd.concat([baseline_results, pd.DataFrame(comparison_rows)], ignore_index=True)
comparison = comparison.sort_values("accuracy_mean", ascending=False).reset_index(drop=True)
comparison[["model", "accuracy_mean", "accuracy_std", "roc_auc_mean"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#e45756" if m == "Stacking Ensemble" else ACCENT for m in comparison["model"]]
ax.barh(comparison["model"][::-1], comparison["accuracy_mean"][::-1],
        xerr=comparison["accuracy_std"][::-1], color=colors[::-1], capsize=4)
ax.set_xlim(max(0, comparison["accuracy_mean"].min() - 0.05), 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Accuracy: individual models vs. stacking ensemble", pad=12, fontweight="bold")
ax.set_xlabel("Mean CV accuracy")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Final training and submission

In [ ]:
best_single_name = baseline_results.loc[0, "model"]
best_single_acc = baseline_results.loc[0, "accuracy_mean"]
stack_acc = stack_cv["test_accuracy"].mean()

if stack_acc >= best_single_acc:
    final_pipeline = stacking_pipeline
    final_name = "Stacking Ensemble"
else:
    final_pipeline = build_pipeline(tuned_models.get(best_single_name, build_models()[best_single_name]))
    final_name = best_single_name

print(f"Final model selected for submission: {final_name}")

final_pipeline.fit(X, y)
predictions = final_pipeline.predict(X_test)

submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions.astype(int)})
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")
submission.head()

## 12. Model interpretation: feature importance

SHAP needs a single tree model, so interpretation uses the best individually
tuned model (even when the stacking ensemble was chosen for the submission).

In [ ]:
interp_name = best_single_name
interp_model = tuned_models.get(interp_name, build_models()[interp_name])
interp_pipeline = build_pipeline(interp_model)
interp_pipeline.fit(X, y)

feature_names = interp_pipeline.named_steps["preprocessor"].get_feature_names_out()
fitted_model = interp_pipeline.named_steps["model"]

if hasattr(fitted_model, "feature_importances_"):
    importance_values = fitted_model.feature_importances_
    importance_label = "Importance"
else:
    importance_values = np.abs(fitted_model.coef_).ravel()
    importance_label = "Absolute coefficient"

feature_importance = (
    pd.DataFrame({"Feature": feature_names, importance_label: importance_values})
    .sort_values(importance_label, ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=feature_importance, x=importance_label, y="Feature", color=ACCENT_2)
ax.set_title(f"Top 15 features — {interp_name}", pad=12, fontweight="bold")
ax.set_xlabel(importance_label)
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

feature_importance

## 13. Model interpretation: SHAP

SHAP (SHapley Additive exPlanations) values quantify each feature's
contribution to individual predictions, giving a more nuanced picture than
global feature importance alone.

In [ ]:
X_transformed = interp_pipeline.named_steps["preprocessor"].transform(X)
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), size=min(300, len(X)), replace=False)
X_shap_sample = X_transformed[sample_idx]

explainer = shap.TreeExplainer(fitted_model)
shap_values = explainer.shap_values(X_shap_sample)
if isinstance(shap_values, list):  # some binary classifiers return [class0, class1]
    shap_values = shap_values[1]

shap.summary_plot(shap_values, X_shap_sample, feature_names=feature_names, max_display=15, show=False)
plt.title(f"SHAP summary — {interp_name}", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = (
    pd.DataFrame({"Feature": feature_names, "Mean |SHAP value|": mean_abs_shap})
    .sort_values("Mean |SHAP value|", ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=shap_importance, x="Mean |SHAP value|", y="Feature", color="#e45756")
ax.set_title(f"Mean |SHAP value| by feature — {interp_name}", pad=12, fontweight="bold")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

shap_importance